In [1]:
from surprise import Dataset, SVD, Reader
from surprise.model_selection import cross_validate

In [2]:
import pandas as pd
import scipy.sparse as sp
from scipy.sparse import csr_matrix
import numpy as np
from implicit.als import AlternatingLeastSquares
import os
import time
from typing import List, Tuple
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

In [3]:
def split_data_by_time(data,test_size = 0.3):
  """Split data by timestamp

  Args:
      data (pd.DataFrame): The data to split
      test_size (float, optional): The size of the test set. Defaults to 0.3.
  """
  data = data.sort_values(by='timestamp')
  threshold = int(data.shape[0] * test_size)
  return data.iloc[:-threshold], data.iloc[-threshold:]

movies = pd.read_csv('ml-20m/movies.csv')
ratings = pd.read_csv('ml-20m/ratings.csv')
ratings.sort_values('timestamp',inplace = True)

user_encoder,item_encoder = LabelEncoder(),LabelEncoder()
ratings['userId'] = user_encoder.fit_transform(ratings['userId'])
ratings['movieId'] = item_encoder.fit_transform(ratings['movieId'])

df_train,df_test = split_data_by_time(ratings,0.2)
df_cand, df_rank = split_data_by_time(df_train,0.5)
df_rank = df_rank[df_rank.userId.isin(df_cand.userId.unique())]

In [16]:
def get_svd_preds(data, user_ids = None,item_ids = None, N=100, filter_already_watched = True) -> pd.DataFrame:
    reader = Reader(rating_scale=(1, 5))
    rating_surprise = Dataset.load_from_df(data[['userId', 'movieId', 'rating']], reader)
    trainset = rating_surprise.build_full_trainset()
    print()
    model = SVD()
    model.fit(trainset)

    top_n_pred = dict()
    
    users_to_predict = data['userId'].unique()

    items = np.arange(data['movieId'].max())
    
    if not type(user_ids) == None:
        users_to_predict = user_ids

    for user in users_to_predict:
        print(user)
        preds_for_user = []
        cur_items = np.setdiff1d(items,data[data['userId'] == user]['movieId'].values)
        
        for item in items:
            pred = model.predict(user,item,None)
            preds_for_user.append((pred.iid,pred.est))

        top_n_pred[user] = sorted(preds_for_user, key = lambda x: x[1], reverse = True)[:N]
        

    result = pd.DataFrame({'userId': [i for i in users_to_predict],
                            'movieId': [[i[0] for i in top_n_pred[user]] for user in users_to_predict],
                            'svd_score': [[i[1] for i in top_n_pred[user]] for user in users_to_predict]})
    
    return result.explode(['movieId','svd_score'])

In [ ]:
df_pred_svd = get_svd_preds(df_train,df_test.userId.unique(),df_test.movieId.unique(),50)

In [17]:
def recall_at_k(df_true, df_pred, k=40, threshold=3.5):
    return (
        df_true[df_true['rating'] >= threshold]
        .merge(
            df_pred.sort_values('svd_score', ascending=False)
            .groupby('userId')
            .head(k)
            .assign(is_valid=1)[['userId', 'movieId', 'is_valid']],
            how='left',
            on=['userId', 'movieId']
        )
        .fillna(0)
        .groupby('userId')
        .apply(lambda x: x['is_valid'].sum() / len(x))
        .mean()
    )

In [20]:
recall_at_k(df_test,df_pred_svd,50,4)

C:\Users\vova-\AppData\Local\Temp\ipykernel_21972\4223491934.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(0)
C:\Users\vova-\AppData\Local\Temp\ipykernel_21972\4223491934.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x['is_valid'].sum() / len(x))


0.11535845631938897

In [25]:
df_pred = df_pred_svd

In [ ]:
df_ranker_pred = get_svd_preds(df_cand,df_rank.userId.unique(),N = 100)

In [27]:
df_ranker_pred

,userId,movieId,svd_score
0,115351,2706,4.721035
0,115351,2028,4.60052
0,115351,1923,4.57667
0,115351,3916,4.574179
0,115351,318,4.550836
...,...,...,...
4257,27935,2176,4.77561
4257,27935,1260,4.772123
4257,27935,1136,4.770593
4257,27935,1253,4.769472


In [29]:
tags = pd.read_csv('ml-20m/tags.csv')
tags = tags.groupby(['movieId','tag']).size().reset_index(name='count')
most_pop_tag  = tags.loc[tags.groupby('movieId')['count'].idxmax()]

def get_tag(df):
    return df.merge(most_pop_tag[['movieId','tag']],on = 'movieId',how = 'left').fillna('unknown')
df_ranker_pred  = get_tag(df_ranker_pred)
df_ranker_pred

C:\Users\vova-\AppData\Local\Temp\ipykernel_21972\1738603709.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.merge(most_pop_tag[['movieId','tag']],on = 'movieId',how = 'left').fillna('unknown')


,userId,movieId,svd_score,tag
0,115351,2706,4.721035,comedy
1,115351,2028,4.600520,World War II
2,115351,1923,4.576670,Ben Stiller
3,115351,3916,4.574179,football
4,115351,318,4.550836,prison
...,...,...,...,...
425795,27935,2176,4.775610,Alfred Hitchcock
425796,27935,1260,4.772123,Fritz Lang
425797,27935,1136,4.770593,Monty Python
425798,27935,1253,4.769472,aliens


In [30]:
from catboost import CatBoost, Pool

def get_pool(df, features, cat_features = None):
  df = df.sort_values(by = ['userId','movieId'])

  group_id = df['userId'].values
  y = df['rating'].values

  pool = Pool(data = df[features],
              group_id = group_id,
              label = y,
              cat_features = cat_features,
              feature_names = features
              )
  return pool

In [31]:
all_users = df_ranker_pred['userId'].unique()
np.random.shuffle(all_users)
threshold = int(0.25*len(all_users))
eval_users,test_users,train_users, = all_users[:threshold],all_users[threshold:threshold*2],all_users[2*threshold:]

In [37]:
df_ranker_pred = df_ranker_pred.merge(df_rank[['userId','movieId','rating']],on = ['movieId','userId'],
                                     how = 'left')

In [43]:
df_ranker_pred.rating = df_ranker_pred.rating.fillna(0)

In [44]:
eval_pool = get_pool(df_ranker_pred[df_ranker_pred['userId'].isin(eval_users)],
                    ['userId','movieId','svd_score','tag'],['tag'])
test_pool = get_pool(df_ranker_pred[df_ranker_pred['userId'].isin(test_users)],
                    ['userId','movieId','svd_score','tag'],['tag'])
train_pool = get_pool(df_ranker_pred[df_ranker_pred['userId'].isin(train_users)],
                     ['userId','movieId','svd_score','tag'],['tag'])

In [45]:
params = {'boosting_type':'Plain',
          'devices':'0',
          'early_stopping_rounds':100,
          'eval_metric':'RecallAt:top=50',
          'learning_rate':0.1,
          'max_ctr_complexity':1,
          'nan_mode':'Min',
          'num_trees':2000,
          'objective':'PairLogitPairwise',
          'random_state':52,
          'task_type':'GPU',
          'one_hot_max_size':30
          }
ranker = CatBoost(params = params)
ranker.fit(X=train_pool,metric_period=100,eval_set = eval_pool)

Metric RecallAt:top=50 is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.6114255	test: 0.6054185	best: 0.6054185 (0)	total: 216ms	remaining: 7m 12s
100:	learn: 0.7550149	test: 0.7452109	best: 0.7452109 (100)	total: 5.53s	remaining: 1m 43s
200:	learn: 0.7734148	test: 0.7519512	best: 0.7529602 (184)	total: 10.7s	remaining: 1m 35s
300:	learn: 0.7857676	test: 0.7531326	best: 0.7540492 (277)	total: 15.8s	remaining: 1m 29s
400:	learn: 0.7948768	test: 0.7529119	best: 0.7544154 (321)	total: 20.9s	remaining: 1m 23s
500:	learn: 0.8012515	test: 0.7537756	best: 0.7564692 (427)	total: 25.9s	remaining: 1m 17s
bestTest = 0.7564691911
bestIteration = 427
Shrink model to first 428 iterations.


In [46]:
ranker.eval_metrics(test_pool, metrics=["RecallAt:top=50"],ntree_start=ranker.tree_count_-1 )

{'RecallAt:top=50': [0.7500934395929572]}

In [49]:
df_pred = get_tag(df_pred)
df_pred

C:\Users\vova-\AppData\Local\Temp\ipykernel_21972\1738603709.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.merge(most_pop_tag[['movieId','tag']],on = 'movieId',how = 'left').fillna('unknown')


,userId,movieId,svd_score,tag
0,6099,7502,4.781612,World War II
1,6099,7767,4.441665,beautiful
2,6099,38159,4.421424,loneliness
3,6099,32657,4.414950,beautiful
4,6099,31547,4.408862,Werner Herzog
...,...,...,...,...
1583495,136690,904,4.083003,Alfred Hitchcock
1583496,136690,2571,4.081284,sci-fi
1583497,136690,26712,4.081129,7n Up (series)
1583498,136690,47152,4.073928,boring


In [50]:
df_pred['ranker_score'] = ranker.predict(df_pred)

In [54]:
def recall_at_k(df_true, df_pred, k=40, threshold=4):
    return (
        df_true[df_true['rating'] >= threshold]
        .merge(
            df_pred.sort_values('ranker_score', ascending=False)
            .groupby('userId')
            .head(k)
            .assign(is_valid=1)[['userId', 'movieId', 'is_valid']],
            how='left',
            on=['userId', 'movieId']
        )
        .fillna(0)
        .groupby('userId')
        .apply(lambda x: x['is_valid'].sum() / len(x))
        .mean()
    )

In [55]:
recall_at_k(df_test,df_pred)

C:\Users\vova-\AppData\Local\Temp\ipykernel_21972\592241265.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x['is_valid'].sum() / len(x))


0.1104552807221732

In [57]:
def get_svd_predsss(data, user_ids = None,item_ids = None, N=100, filter_already_watched = True) -> pd.DataFrame:
    reader = Reader(rating_scale=(1, 5))
    rating_surprise = Dataset.load_from_df(data[['userId', 'movieId', 'rating']], reader)
    trainset = rating_surprise.build_full_trainset()
    print()
    model = SVD()
    model.fit(trainset)
    return model

In [58]:
model = get_svd_predsss(df_train,N = 50)

In [59]:
from surprise.dump import dump
dump('cand_model',algo = model)
ranker.save_model("ranker_model.cbm")

In [2]:
import implicit

In [2]:
!wget http://files.grouplens.org/datasets/movielens/ml-20m.zip
!unzip ml-20m.zip

"wget" ­Ґ пў«пҐвбп ў­гваҐ­­Ґ© Ё«Ё ў­Ґи­Ґ©
Є®¬ ­¤®©, ЁбЇ®«­пҐ¬®© Їа®Ја ¬¬®© Ё«Ё Ї ЄҐв­л¬ д ©«®¬.
"unzip" ­Ґ пў«пҐвбп ў­гваҐ­­Ґ© Ё«Ё ў­Ґи­Ґ©
Є®¬ ­¤®©, ЁбЇ®«­пҐ¬®© Їа®Ја ¬¬®© Ё«Ё Ї ЄҐв­л¬ д ©«®¬.


In [2]:
import urllib.request
import zipfile
import os

url = "http://files.grouplens.org/datasets/movielens/ml-20m.zip"
zip_path = "ml-20m.zip"

# скачать
urllib.request.urlretrieve(url, zip_path)

# распаковать
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(".")

print("Готово")


Готово


In [3]:
import pandas as pd
import scipy.sparse as sp
from scipy.sparse import csr_matrix
import numpy as np
from implicit.als import AlternatingLeastSquares
import os
import time
from typing import List, Tuple
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

In [8]:
movies = pd.read_csv('ml-20m/movies.csv')
ratings = pd.read_csv('ml-20m/ratings.csv')

In [ ]:
sns.histplot(ratings['rating'])

In [ ]:
sns.histplot(ratings.groupby('movieId').size(),log_scale=True)

In [9]:
ratings.sort_values('timestamp',inplace = True)
ratings.head(5)

,userId,movieId,rating,timestamp
4182421,28507,1176,4.0,789652004
18950979,131160,1079,3.0,789652009
18950936,131160,47,5.0,789652009
18950930,131160,21,3.0,789652009
12341178,85252,45,3.0,822873600


In [10]:

def get_tag(df):
    return df.merge(most_pop_tag[['movieId','tag']],on = 'movieId',how = 'left').fillna('unknown')
df_cand_preds = get_tag(df_cand_preds)

## Splitting data to train and test

In [11]:
def split_data_by_user(data,test_size = 0.3):
  """Last n interaction of each user is used as test data

  Args:
      data (pd.DataFrame): The data to split
      test_size (float, optional): The size of the test set. Defaults to 0.3.
  """
  train_data,test_data = [],[]
  data = data.sort_values(by='timestamp')

  for user_id, user_data in data.groupby('userId'):
    threshold = int(data.shape[0] * test_size)
    train_data.append(user_data.iloc[:-threshold])
    test_data.append(user_data.iloc[-threshold:])


  train_data = pd.concat(train_data)
  test_data = pd.concat(test_data)
  return train_data,test_data


def split_data_by_time(data,test_size = 0.3):
  """Split data by timestamp

  Args:
      data (pd.DataFrame): The data to split
      test_size (float, optional): The size of the test set. Defaults to 0.3.
  """
  data = data.sort_values(by='timestamp')
  threshold = int(data.shape[0] * test_size)
  return data.iloc[:-threshold], data.iloc[-threshold:]

In [12]:
df_train,df_test = split_data_by_time(ratings,0.2)
df_cand, df_rank = split_data_by_time(df_train,0.5)
df_rank = df_rank[df_rank.userId.isin(df_cand.userId.unique())]

In [13]:
df_cand.shape,df_rank.shape,df_test.shape

((8000106, 4), (913393, 4), (4000052, 4))

In [14]:
# пересечений нет
df_test.merge(df_train[['movieId','userId']], on = ['movieId','userId'],how = 'inner')

,userId,movieId,rating,timestamp


In [15]:
df_test = df_test[(df_test['rating']>=3.5) & 
    df_test['userId'].isin(df_train['userId'].unique()) & 
    df_test['movieId'].isin(df_train['movieId'].unique())]
df_test.shape

(312500, 4)

## ALS

In [152]:
def get_als_pred(ratings, user_to_pred, N):

    user_ids = ratings['userId'].unique().tolist()
    item_ids = ratings['movieId'].unique().tolist()
    
    user_id_to_index = {user_id: idx for idx, user_id in enumerate(user_ids)}
    item_id_to_index = {item_id: idx for idx, item_id in enumerate(item_ids)}
    index_to_item_id = {v:k for k,v in item_id_to_index.items()}

    rows = ratings['userId'].map(user_id_to_index).to_list()
    cols = ratings['movieId'].map(item_id_to_index).to_list()
    values = ratings['rating'].to_list()

    print('users len:    ', len(user_ids), 'item len:    ',len(item_ids))
    sparse_matrix = csr_matrix((values,(rows,cols)),shape = (len(user_ids),len(item_ids)))

    model = AlternatingLeastSquares(factors = 50, iterations = 15,alpha = 1)
    model.fit(sparse_matrix)

    user_to_pred = np.array([user_id_to_index[i] for i in user_to_pred])

    rec_indxs, scores = model.recommend(user_to_pred,
                                        sparse_matrix[user_to_pred], N = N,
                                        filter_already_liked_items = True)

    df_preds = pd.DataFrame({
        'userId':list(user_to_pred),
        'movieId':[
                [index_to_item_id[i] for i in i] for i in rec_indxs.tolist()
            ],
        'als_score': scores.tolist()
        
    })
    df_preds = df_preds.explode(['movieId','als_score'])
    return df_preds

In [153]:
df_cand_preds = get_als_pred(df_cand,df_rank.userId.unique(),400)

users len:     71694 item len:     6050


  0%|          | 0/15 [00:00<?, ?it/s]

In [154]:
df_pred = get_als_pred(df_train,df_test.userId.unique(),400)

users len:     112466 item len:     12388


  0%|          | 0/15 [00:00<?, ?it/s]

In [101]:
df_cand_preds = pd.merge(df_cand_preds, ratings[['movieId','userId','rating']],
                         on=['userId','movieId'],how='left').fillna(0)

C:\Users\vova-\AppData\Local\Temp\ipykernel_14340\1117942619.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  on=['userId','movieId'],how='left').fillna(0)


### todo: прикрутить фичи к выходу als(df_cand_preds)

## Ranking

In [102]:
tags = pd.read_csv('ml-20m/tags.csv')
tags = tags.groupby(['movieId','tag']).size().reset_index(name='count')
most_pop_tag  = tags.loc[tags.groupby('movieId')['count'].idxmax()]

In [103]:
def get_tag(df):
    return df.merge(most_pop_tag[['movieId','tag']],on = 'movieId',how = 'left').fillna('unknown')
df_cand_preds = get_tag(df_cand_preds)

In [106]:
from catboost import CatBoost, Pool

In [107]:
def get_pool(df, features, cat_features = None):
  df = df.sort_values(by = ['userId','movieId'])

  group_id = df['userId'].values
  y = df['rating'].values

  pool = Pool(data = df[features],
              group_id = group_id,
              label = y,
              cat_features = cat_features,
              feature_names = features
              )
  return pool

In [108]:
all_users = df_cand_preds['userId'].unique()
np.random.shuffle(all_users)
threshold = int(0.25*len(all_users))
eval_users,test_users,train_users, = all_users[:threshold],all_users[threshold:threshold*2],all_users[2*threshold:]

In [109]:
eval_pool = get_pool(df_cand_preds[df_cand_preds['userId'].isin(eval_users)],
                    ['userId','movieId','als_score','tag'],['tag'])
test_pool = get_pool(df_cand_preds[df_cand_preds['userId'].isin(test_users)],
                    ['userId','movieId','als_score','tag'],['tag'])
train_pool = get_pool(df_cand_preds[df_cand_preds['userId'].isin(train_users)],
                     ['userId','movieId','als_score','tag'],['tag'])

In [114]:
params = {'boosting_type':'Plain',
          'devices':'0',
          'early_stopping_rounds':100,
          'eval_metric':'RecallAt:top=50',
          'learning_rate':0.1,
          'max_ctr_complexity':1,
          'nan_mode':'Min',
          'num_trees':2000,
          'objective':'PairLogitPairwise',
          'random_state':52,
          'task_type':'GPU',
          'one_hot_max_size':30
          }
ranker = CatBoost(params = params)

In [115]:
ranker.fit(X=train_pool,metric_period=100,eval_set = eval_pool)

Metric RecallAt:top=50 is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.2646848	test: 0.2756432	best: 0.2756432 (0)	total: 899ms	remaining: 29m 56s
100:	learn: 0.3722705	test: 0.3722045	best: 0.3724175 (99)	total: 41.9s	remaining: 13m 7s
200:	learn: 0.3840694	test: 0.3774404	best: 0.3784627 (193)	total: 1m 21s	remaining: 12m 5s
300:	learn: 0.3884612	test: 0.3769400	best: 0.3785248 (279)	total: 2m	remaining: 11m 18s
400:	learn: 0.3924763	test: 0.3788304	best: 0.3792785 (387)	total: 2m 38s	remaining: 10m 33s
500:	learn: 0.3942266	test: 0.3805372	best: 0.3813886 (470)	total: 3m 17s	remaining: 9m 51s
600:	learn: 0.3962128	test: 0.3802923	best: 0.3815993 (503)	total: 3m 55s	remaining: 9m 8s
bestTest = 0.3815992899
bestIteration = 503
Shrink model to first 504 iterations.


In [117]:
ranker.eval_metrics(test_pool, metrics=["RecallAt:top=50"],ntree_start=ranker.tree_count_-1 )

{'RecallAt:top=50': [0.3733153489536211]}

In [141]:
df_cand_preds

,userId,movieId,als_score,rating,tag
0,69343,1242,1.179761,0.0,Civil War
1,69343,3894,1.151548,0.0,Benito Zambrano
2,69343,5352,1.103845,0.0,DVD-RAM
3,69343,4226,1.099165,0.0,nonlinear
4,69343,2615,1.091542,0.0,80s nostalgia
...,...,...,...,...,...
1703195,69057,4633,0.408273,0.0,black comedy
1703196,69057,3859,0.407813,0.0,DVD-Video
1703197,69057,2910,0.407272,0.0,nudity (full frontal)
1703198,69057,1059,0.405954,0.0,Leonardo DiCaprio


In [ ]:
df_cand_preds['cb_rating'] = ranker.predict(df_cand_preds)

### Try to add tags

In [75]:
# df_cand_preds_labels = df_cand_preds.copy()
# df_cand_preds_labels.rating = df_cand_preds_labels.rating.apply(lambda x: 0 if x<=3.5 else 1)
# df_test_labels = df_cand_preds[df_cand_preds['userId'].isin(eval_users)].copy()
# df_test_labels.rating = df_test_labels.rating.apply(lambda x: 0 if x<=3.5 else 1)


In [76]:
tags = pd.read_csv('ml-20m/tags.csv')

In [77]:
tags['tag'] = LabelEncoder().fit_transform(tags['tag'])

In [78]:
df_cand_preds_tags = pd.merge(df_cand_preds,tags[['movieId','tag']],
         on = ['movieId'],how = 'inner').drop_duplicates(['movieId','userId'])

In [79]:
df_cand_preds_tags

,userId,movieId,als_score,rating,tag
0,69343,220,1.247209,0.0,6212
3,69343,38,1.058681,0.0,37038
8,69343,2417,1.047786,0.0,20768
12,69343,3320,0.997856,0.0,326
16,69343,3702,0.897645,0.0,23855
...,...,...,...,...,...
42658711,69057,4141,0.205267,0.0,10949
42658715,69057,4896,0.202544,0.0,18515
42658994,69057,3899,0.201878,0.0,22646
42658995,69057,2585,0.200843,0.0,22404


In [80]:
all_users = df_cand_preds_tags['userId'].unique()
np.random.shuffle(all_users)
threshold = int(0.2*len(all_users))
eval_users,test_users,train_users, = all_users[:threshold],all_users[threshold:threshold*2],all_users[2*threshold:]
eval_pool = get_pool(df_cand_preds_tags[df_cand_preds_tags['userId'].isin(eval_users)],
                     ['userId','movieId','als_score','tag'],['tag'])
test_pool = get_pool(df_cand_preds_tags[df_cand_preds_tags['userId'].isin(test_users)],
                     ['userId','movieId','als_score','tag'],['tag'])
train_pool = get_pool(df_cand_preds_tags[df_cand_preds_tags['userId'].isin(train_users)],
                      ['userId','movieId','als_score','tag'],['tag'])

new_ranker = CatBoost(params = params)
new_ranker.fit(X=train_pool,metric_period=100,eval_set = eval_pool)

Groupwise loss function. OneHotMaxSize set to 10


Metric RecallAt:top=50 is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.4794314	test: 0.4641471	best: 0.4641471 (0)	total: 252ms	remaining: 8m 24s
100:	learn: 0.5982598	test: 0.5868908	best: 0.5868908 (100)	total: 12.4s	remaining: 3m 53s
200:	learn: 0.6105168	test: 0.5959173	best: 0.5965405 (179)	total: 24.5s	remaining: 3m 38s
300:	learn: 0.6179891	test: 0.5993012	best: 0.6010848 (296)	total: 36.3s	remaining: 3m 25s
400:	learn: 0.6239902	test: 0.6002205	best: 0.6023429 (349)	total: 48.2s	remaining: 3m 12s
bestTest = 0.6023428699
bestIteration = 349
Shrink model to first 350 iterations.


In [83]:
new_ranker.eval_metrics(test_pool, metrics=["RecallAt:top=50"],ntree_start=new_ranker.tree_count_-1 )

{'RecallAt:top=50': [0.6165743897625702]}

### increase one hot size param cotboost

In [79]:
one_hot_params = {'boosting_type':'Plain',
          'devices':'0',
          'early_stopping_rounds':100,
          'eval_metric':'RecallAt:top=50',
          'learning_rate':0.1,
          'max_ctr_complexity':1,
          'nan_mode':'Min',
          'num_trees':2000,
          'objective':'PairLogitPairwise',
          'random_state':52,
          'task_type':'GPU',
          'one_hot_max_size':30
          }
one_hot_model = CatBoost(params = one_hot_params)
one_hot_model.fit(X = train_pool,metric_period = 100, eval_set = eval_pool)

Metric RecallAt:top=50 is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.5203815	test: 0.5180333	best: 0.5180333 (0)	total: 242ms	remaining: 8m 3s
100:	learn: 0.5950691	test: 0.6019362	best: 0.6019362 (100)	total: 13.2s	remaining: 4m 7s
200:	learn: 0.6080412	test: 0.6094254	best: 0.6096151 (198)	total: 26.4s	remaining: 3m 56s
300:	learn: 0.6139626	test: 0.6125326	best: 0.6152045 (268)	total: 38.9s	remaining: 3m 39s
400:	learn: 0.6185146	test: 0.6167246	best: 0.6167246 (400)	total: 51.4s	remaining: 3m 25s
500:	learn: 0.6216119	test: 0.6185330	best: 0.6191895 (489)	total: 1m 4s	remaining: 3m 12s
600:	learn: 0.6258906	test: 0.6163891	best: 0.6194394 (539)	total: 1m 17s	remaining: 3m
bestTest = 0.6194394378
bestIteration = 539
Shrink model to first 540 iterations.


### Try to add genres

In [84]:
movies['genres'] = LabelEncoder().fit_transform(movies['genres'])
df_cand_preds_gen = pd.merge(df_cand_preds_tags,movies[['movieId','genres']],
                             on = 'movieId',how = 'inner').drop_duplicates(['userId','movieId'])

In [86]:
one_hot_params = {'boosting_type':'Plain',
          'devices':'0',
          'early_stopping_rounds':100,
          'eval_metric':'RecallAt:top=50',
          'learning_rate':0.1,
          'max_ctr_complexity':1,
          'nan_mode':'Min',
          'num_trees':2000,
          'objective':'PairLogitPairwise',
          'random_state':52,
          'task_type':'GPU',
          'one_hot_max_size':30
          }
all_users = df_cand_preds_gen['userId'].unique()
np.random.shuffle(all_users)
threshold = int(0.2*len(all_users))
eval_users,test_users,train_users, = all_users[:threshold],all_users[threshold:threshold*2],all_users[2*threshold:]
eval_pool = get_pool(df_cand_preds_gen[df_cand_preds_gen['userId'].isin(eval_users)],
                     ['userId', 'movieId', 'als_score', 'tag', 'genres'],['tag','genres'])
test_pool = get_pool(df_cand_preds_gen[df_cand_preds_gen['userId'].isin(test_users)],
                     ['userId', 'movieId', 'als_score', 'tag', 'genres'],['tag','genres'])
train_pool = get_pool(df_cand_preds_gen[df_cand_preds_gen['userId'].isin(train_users)],
                     ['userId', 'movieId', 'als_score', 'tag', 'genres'],['tag','genres'])

tags_gen_ranker = CatBoost(params = one_hot_params)
tags_gen_ranker.fit(X=train_pool,metric_period=100,eval_set = eval_pool)

Metric RecallAt:top=50 is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.4931812	test: 0.4865892	best: 0.4865892 (0)	total: 243ms	remaining: 8m 6s
100:	learn: 0.6241371	test: 0.6155674	best: 0.6158674 (99)	total: 12.2s	remaining: 3m 50s
200:	learn: 0.6414687	test: 0.6306819	best: 0.6307873 (194)	total: 24.2s	remaining: 3m 36s
300:	learn: 0.6500195	test: 0.6319866	best: 0.6359870 (289)	total: 35.9s	remaining: 3m 22s
400:	learn: 0.6556742	test: 0.6380749	best: 0.6380749 (400)	total: 47.6s	remaining: 3m 9s
500:	learn: 0.6599809	test: 0.6385812	best: 0.6399624 (483)	total: 59.3s	remaining: 2m 57s
600:	learn: 0.6657325	test: 0.6394121	best: 0.6416061 (599)	total: 1m 10s	remaining: 2m 45s
700:	learn: 0.6705369	test: 0.6445803	best: 0.6447731 (694)	total: 1m 22s	remaining: 2m 32s
800:	learn: 0.6750486	test: 0.6444002	best: 0.6454877 (744)	total: 1m 34s	remaining: 2m 20s
bestTest = 0.6454877026
bestIteration = 744
Shrink model to first 745 iterations.


In [87]:
tags_gen_ranker.eval_metrics(test_pool, metrics=["RecallAt:top=40"],ntree_start=tags_gen_ranker.tree_count_-1 )

{'RecallAt:top=40': [0.566721508374094]}

## Calc eval metric

In [89]:
df_pred = get_als_pred(df_train,df_test['userId'].unique(),N = 200)

users len:     112466 item len:     12388


  0%|          | 0/10 [00:00<?, ?it/s]

In [90]:
tags = tags.drop_duplicates('movieId')

In [91]:
df_pred = pd.merge(df_pred,tags[['movieId','tag']],
         on = ['movieId'],how = 'inner').drop_duplicates(['movieId','userId'])
df_pred = pd.merge(df_pred,movies[['movieId','genres']],
                             on = 'movieId',how = 'inner').drop_duplicates(['userId','movieId'])
df_pred['catboost_score'] = new_ranker.predict(df_pred)

In [92]:
df_pred.sort_values('catboost_score',ascending =False).groupby('userId').head(10)

,userId,movieId,als_score,tag,genres,catboost_score
432515,44195,315,1.049012,12410,395,3.424515
612424,46172,315,0.969129,12410,395,3.341516
416101,56458,315,1.15822,12410,395,3.275855
665699,43920,315,0.920766,12410,395,3.272861
83466,41430,315,1.111363,12410,395,3.271408
...,...,...,...,...,...,...
758743,106569,1587,0.093158,18452,135,-0.267845
758755,106569,1317,0.080247,3045,893,-0.326229
520370,111929,4548,0.054064,3045,304,-0.392351
520376,111929,6427,0.052099,28344,862,-0.514088


In [93]:
def recall_at_k(df_true, df_pred, k = 40, treshold = 3.5):
    df_true = df_true[df_true.rating >= treshold].copy()
    df_pred = df_pred.sort_values('catboost_score', ascending =False).groupby('userId').head(k).copy()
    df_pred['is_valid'] = [1 for i in range(len(df_pred))]
    df_true = df_true.merge(df_pred,how='left',on = ['userId','movieId']).fillna(0).groupby('userId')
    

In [142]:
def recall_at_k_short(df_true, df_pred, k=40, threshold=3.5):
    """Сокращенная версия функции"""
    return (
        df_true[df_true['rating'] >= threshold]
        .merge(
            df_pred.sort_values('catboost_score', ascending=False)
            .groupby('userId')
            .head(k)
            .assign(is_valid=1)[['userId', 'movieId', 'is_valid']],
            how='left',
            on=['userId', 'movieId']
        )
        .fillna(0)
        .groupby('userId')
        .apply(lambda x: x['is_valid'].sum() / len(x))
        .mean()
    )

In [146]:
def recall_at_k(df_true, df_pred, k=40, threshold=3.5):
    """Сокращенная версия функции"""
    return (
        df_true[df_true['rating'] >= threshold]
        .merge(
            df_pred.sort_values('als_score', ascending=False)
            .groupby('userId')
            .head(k)
            .assign(is_valid=1)[['userId', 'movieId', 'is_valid']],
            how='left',
            on=['userId', 'movieId']
        )
        .fillna(0)
        .groupby('userId')
        .apply(lambda x: x['is_valid'].sum() / len(x))
        .mean()
    )

In [167]:
recall_at_k(df_test,df_pred,50)

C:\Users\vova-\AppData\Local\Temp\ipykernel_14340\151995914.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(0)
C:\Users\vova-\AppData\Local\Temp\ipykernel_14340\151995914.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x['is_valid'].sum() / len(x))


np.float64(0.0014796429513315983)

In [160]:
df_pred.shape

(2192400, 3)

In [163]:
df_test.shape[0]/df_test.userId.nunique()

57.015143222039775

In [95]:
recall_at_k_short(df_test,df_pred,40,3.5)

C:\Users\vova-\AppData\Local\Temp\ipykernel_22184\2006697626.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(0)
C:\Users\vova-\AppData\Local\Temp\ipykernel_22184\2006697626.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x['is_valid'].sum() / len(x))


np.float64(0.0012090844277876333)

In [96]:
def precision_at_k_vectorized(df_true, df_pred, k=40, threshold=3.5):
    """
    Векторизованная версия для максимальной производительности
    """
    # Создаем хэш-таблицу релевантных пар
    relevant_set = {
        (uid, mid): True
        for uid, mid in df_true[df_true['rating'] >= threshold][['userId', 'movieId']].values
    }
    
    # Сортируем предсказания
    df_pred_sorted = df_pred.sort_values(['userId', 'catboost_score'], ascending=[True, False])
    
    # Группируем по пользователям
    user_groups = df_pred_sorted.groupby('userId')
    
    precisions = []
    
    for user_id, group in user_groups:
        # Берем топ-k рекомендаций
        top_k = group.head(k)
        
        # Проверяем релевантность
        relevant_count = sum(
            1 for _, row in top_k.iterrows()
            if (user_id, row['movieId']) in relevant_set
        )
        
        # Вычисляем precision
        precisions.append(relevant_count / k)
    
    return np.mean(precisions) if precisions else 0.0

In [97]:
precision_at_k_vectorized(df_test,df_pred,40,3)

np.float64(0.001509760992519613)

In [ ]:
# from sqlalchemy import create_engine

# engine = create_engine('sqlite:///example.db', echo=True)
# ratings.to_sql('ratings', con=engine, if_exists='replace', index=False)
# movies.to_sql('movies', con=engine, if_exists='replace', index=False)

In [ ]:
import sqlite3
con = sqlite3.connect("example.db")
cur = con.cursor()
res = cur.execute("SELECT * FROM ratings LIMIT 5")
res.fetchall()

In [ ]:
res = cur.execute("SELECT * FROM ratings LIMIT 5")
res.fetchall()
# pd.DataFrame(res.fetchall(),columns=['movieId','title','genres'])

In [ ]:
def split_data(
    ratings: pd.DataFrame,
) -> Tuple[sp._csr.csr_matrix, List[Tuple[float, float, float]]]:
    """Splitting dataset into train and test

    Args:
        ratings (pd.DataFrame): The ratings data

    Returns:
        Tuple[sp._csr.csr_matrix, List[Tuple[float, float, float]]]: The train and test datasets
    """
    user_item_matrix = ratings.pivot(
        index="userId", columns="movieId", values="rating"
    ).fillna(0)

    # Convert to sparse matrix
    ratings_csr = csr_matrix(user_item_matrix.values)

    # Select 20% of users randomly
    np.random.seed(52)
    unique_users = ratings["userId"].unique()
    test_users = np.random.choice(
        unique_users, size=int(0.5 * len(unique_users)), replace=False
    )

    # Prepare train and test data
    train_data = ratings_csr.copy().toarray()
    test_data = []

    for user in test_users:
        user_index = np.where(unique_users == user)[0][0]
        user_ratings = np.where(train_data[user_index] > 0)[0]
        if len(user_ratings) > 0:
            test_item = np.random.choice(user_ratings)
            test_data.append((user_index, test_item, train_data[user_index, test_item]))
            train_data[user_index, test_item] = 0  # Remove one rating

    train_data = csr_matrix(train_data)
    return train_data, test_data

In [ ]:
train_data, test_data = split_data(ratings)

In [ ]:
train_data, len(test_data)

In [ ]:
model = AlternatingLeastSquares(factors=10, regularization=0.5, alpha=10,iterations=15)
model.fit(coo_matrix(ratings))

In [111]:
import joblib
joblib.dump(new_ranker, 'model_ranker.pkl')

['model_ranker.pkl']

In [ ]:
download_model = joblib.load('model.pkl')
download_model.recommend(0,csr_utils[0],10,filter_already_liked_items=False)

In [ ]:
def mse(item_emb,user_emb,test_data):
  test_predictions = []

  for user_index, item_index, true_rating in test_data:
      predicted_rating = np.dot(user_emb[user_index], item_emb[item_index])
      test_predictions.append((user_index, item_index, true_rating, predicted_rating))

  mse = np.mean([(true - pred) ** 2 for (_, _, true, pred) in test_predictions])
  return mse
def precision_at_k()
print(f"Test MSE: {mse(model.item_factors,model.user_factors,test_data)}")

In [ ]:
item_factors.shape

In [ ]:
test_predictions

In [ ]:
def get_top_k_recommendations_implicit(model, train_data, k=10):
    """Get top-k recommendations for users with implicit library"""
    user_factors = model.user_factors
    item_factors = model.item_factors
    scores = np.dot(user_factors, item_factors.T)
    scores[train_data.nonzero()] = -np.inf  # Exclude already interacted items
    top_k_items = np.argsort(scores, axis=1)[:, -k:][:, ::-1]
    top_k_scores = np.take_along_axis(scores, top_k_items, axis=1)
    return top_k_items, top_k_scores


(
    top_10_recommendations_implicit,
    top_10_scores_implicit,
) = get_top_k_recommendations_implicit(model, train_data, k=10)

In [ ]:
def mrr_at_k(predictions: np.ndarray, ground_truth: List[Tuple[int, int]], k: int = 10):
    """Calculate Mean Reciprocal Rank at K (MRR@K)"""
    mrr = 0.0
    for user_index, true_item in ground_truth:
        top_k_items = predictions[user_index][:k]

        if true_item in top_k_items:
            rank = np.where(top_k_items == true_item)[0][0] + 1
            mrr += 1.0 / rank

    mrr /= len(ground_truth)
    return mrr

In [ ]:
ground_truth = [(user_index, item_index) for user_index, item_index, _ in test_data]
mrr_10_implicit = mrr_at_k(top_10_recommendations_implicit, ground_truth, k=10)
print(f"MRR@10 for `implicit` realization: {mrr_10_implicit}")

In [ ]:
!pip install git+https://github.com/daviddavo/lightfm
import numpy as np
from lightfm.datasets import fetch_movielens
from lightfm import LightFM

In [ ]:
data = fetch_movielens(min_rating=3.0)
print(repr(data['train']))
print(repr(data['test']))

In [ ]:
%time
from scipy.sparse import coo_matrix
model = LightFM(loss='bpr')
model.fit(data['train'], epochs=50, num_threads=2)

In [ ]:
model.user_embeddings @ model.item_embeddings.T

In [ ]:
data['train'].toarray()

In [ ]:
model.item_embeddings.shape

In [ ]:
from lightfm.evaluation import precision_at_k
print("Train precision: %.2f" % precision_at_k(model, coo_matrix(ratings), k=10).mean())
# print("Test precision: %.2f" % precision_at_k(model, coo_matrix(test_data), k=10).mean())

In [ ]:
precision_at_k(model, data['train'], k=10)

In [ ]:
data['train']

In [ ]:
data['item_labels']

In [ ]:
def sample_recommendation(model, data, user_ids):

    n_users, n_items = data['train'].shape

    for user_id in user_ids:
        known_positives = data['item_labels'][data['train'].tocsr()[user_id].indices]

        scores = model.predict(user_id, np.arange(n_items))
        top_items = data['item_labels'][np.argsort(-scores)]

        print("User %s" % user_id)
        print("     Known positives:")

        for x in known_positives[:3]:
            print("        %s" % x)

        print("     Recommended:")

        for x in top_items[:3]:
            print("        %s" % x)

sample_recommendation(model, data, [3, 25, 450])